In [1]:
# imports
import os, glob, math
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.models import load_model
plt.rcParams['figure.figsize'] = (12,5)
print("TF", tf.__version__)


TF 2.20.0


In [2]:
# Data is in Drive, mount it.
from google.colab import drive
drive.mount('/content/drive')

# Set base folder where your dataset is.
BASE_DIR = '/content/drive/MyDrive/major_data'
os.makedirs(BASE_DIR, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

Mounted at /content/drive
BASE_DIR: /content/drive/MyDrive/major_data


In [3]:
folder_path = "/content/drive/MyDrive/major_data"
cleaned_folder = "/content/drive/MyDrive/major_data/cleaned"
os.makedirs(cleaned_folder, exist_ok=True)

required_cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']

# Sort files alphabetically
for file in sorted(os.listdir(folder_path)):

    if not file.lower().endswith(".csv"):
        continue

    try:
        file_path = os.path.join(folder_path, file)
        if not os.path.isfile(file_path):
            print(f" Skipping (not found): {file}")
            continue

        df = pd.read_csv(file_path)

        # -----------------------------
        # Clean headers: strip spaces + fix capitalization
        # -----------------------------
        df.columns = df.columns.str.strip().str.capitalize()

        # -----------------------------
        # Ensure required columns exist
        # -----------------------------
        if not all(col in df.columns for col in required_cols):
            print(f" Missing columns in {file}, skipping.")
            continue

        # Keep only required columns in correct order
        df = df[required_cols]

        # -----------------------------
        # Convert Date & sort
        # -----------------------------
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df = df.dropna(subset=['Date'])
        df = df.sort_values(by='Date')

        # -----------------------------
        # Save cleaned CSV
        # -----------------------------
        save_path = os.path.join(cleaned_folder, file.strip())
        df.to_csv(save_path, index=False)

        print(f" Cleaned & saved: {file}")

    except Exception as e:
        print(f" Error processing {file}: {e}")

 Cleaned & saved: Adani Enterprises Ltd.csv
 Cleaned & saved: Adani ports and special Economic.csv
 Cleaned & saved: Axis Bank Ltd.csv
 Cleaned & saved: Bajaj Finance Ltd.csv
 Cleaned & saved: Bajaj Finserv Ltd.csv
 Cleaned & saved: Bharat Electronics Ltd.csv
 Cleaned & saved: Bharti Airtel Ltd.csv
 Cleaned & saved: Eternal Ltd.csv
 Cleaned & saved: HCL Technologies Ltd.csv
 Cleaned & saved: HDFC Bank Ltd.csv
 Cleaned & saved: Hindustan Unilever Ltd.csv
 Cleaned & saved: ICICI Bank Ltd.csv
 Cleaned & saved: ITC Ltd.csv
 Cleaned & saved: Infosys Ltd.csv
 Cleaned & saved: JSW Steel Ltd.csv
 Cleaned & saved: Kotak Mahindra Bank Ltd.csv
 Cleaned & saved: Larsen and Toubro Ltd.csv
 Cleaned & saved: Mahindra and Mahindra Ltd.csv
 Cleaned & saved: Maruti Suzuki India Ltd.csv
 Cleaned & saved: NTPC Ltd.csv
 Cleaned & saved: Oil and Natural Gas Corporation Ltd.csv
 Cleaned & saved: Power Grid Corporation of India Ltd.csv
 Cleaned & saved: Reliance Industries Ltd.csv
 Cleaned & saved: State Bank

In [4]:
# Use the cleaned folder path
csv_files = glob.glob("/content/drive/MyDrive/major_data/cleaned/*.csv")
print("CSV files found:", csv_files)

processed_data = {}

for file in csv_files:
    company_name = os.path.splitext(os.path.basename(file))[0]
    df = pd.read_csv(file)

    print(f"{company_name} raw shape: {df.shape}")  # Check if file has rows/columns

    # Only keep if df is not empty
    if df.shape[0] > 0:
        processed_data[company_name] = df

print("Companies loaded:", list(processed_data.keys()))

CSV files found: ['/content/drive/MyDrive/major_data/cleaned/Bharti Airtel Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Bajaj Finserv Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Bajaj Finance Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/HDFC Bank Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/HCL Technologies Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Eternal Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Adani ports and special Economic.csv', '/content/drive/MyDrive/major_data/cleaned/Axis Bank Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Bharat Electronics Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Adani Enterprises Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/ICICI Bank Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Larsen and Toubro Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Hindustan Unilever Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/ITC Ltd.csv', '/content/drive/MyDrive/major_data/

In [5]:
# Paths
cleaned_folder = "/content/drive/MyDrive/major_data/cleaned"
csv_files = glob.glob(os.path.join(cleaned_folder, "*.csv"))
print("CSV files found:", csv_files)

# Create output folder
output_folder = "/content/drive/MyDrive/major_data/processed_with_features"
os.makedirs(output_folder, exist_ok=True)

# Processed data dictionary
processed_data = {}

# Loop through CSVs
for file in csv_files:
    company_name = os.path.splitext(os.path.basename(file))[0]
    df = pd.read_csv(file)
    print(f"{company_name} raw shape: {df.shape}")

    # Standardize column names
    df.columns = df.columns.str.strip().str.capitalize()

    # Rename inconsistent columns if needed
    rename_dict = {'Date ': 'Date', 'Close': 'Close', 'Volume': 'Volume'}
    df.rename(columns=rename_dict, inplace=True)

    # Keep only necessary columns
    required_cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
    df = df[[col for col in df.columns if col in required_cols]]

    # Convert numeric columns
    for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')

    # Handle missing data
    df.ffill(inplace=True)
    df.bfill(inplace=True)
    df.dropna(subset=['Open', 'High', 'Low', 'Close', 'Volume'], inplace=True)

    # Convert Date to datetime & sort
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df.dropna(subset=['Date'], inplace=True)
    df = df.sort_values('Date').reset_index(drop=True)

    print(f"{company_name} rows after numeric conversion: {df.shape[0]}")

    # Skip if not enough rows
    if df.shape[0] < 5:  # can adjust threshold based on your feature window
        print(f" Skipping {company_name}, not enough rows")
        continue


    # Feature creation for LSTM
    df['Return'] = df['Close'].pct_change()
    df['High_Low'] = (df['High'] - df['Low']) / df['Low']
    df['Open_Close'] = (df['Open'] - df['Close']) / df['Close']
    df['Volume_Change'] = df['Volume'].pct_change()

    # Drop first row which has NaN from pct_change
    df.dropna(inplace=True)

    # Save processed data
    processed_data[company_name] = df
    print(f" Features created for {company_name}: {df.shape}")

    output_file_path = os.path.join(output_folder, f"{company_name}_features.csv")
    df.to_csv(output_file_path, index=False)
    print(f"Saved processed file with features: {output_file_path}")

# Summary
print("Companies ready:", list(processed_data.keys()))

CSV files found: ['/content/drive/MyDrive/major_data/cleaned/Bharti Airtel Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Bajaj Finserv Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Bajaj Finance Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/HDFC Bank Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/HCL Technologies Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Eternal Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Adani ports and special Economic.csv', '/content/drive/MyDrive/major_data/cleaned/Axis Bank Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Bharat Electronics Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Adani Enterprises Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/ICICI Bank Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Larsen and Toubro Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/Hindustan Unilever Ltd.csv', '/content/drive/MyDrive/major_data/cleaned/ITC Ltd.csv', '/content/drive/MyDrive/major_data/